<a href="https://colab.research.google.com/github/PSD-20/Portfolio-Optimization/blob/main/QAOA_PORTFOLIO_OPTIMIZATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:


import itertools
import numpy as np

# Install Qiskit and Qiskit-Aer if not already installed
%pip install qiskit qiskit-aer

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from scipy.optimize import minimize


# ============================================================
# 1. COLLECT PORTFOLIO DATA FROM THE USER
# ============================================================
#
# Every problem-specific quantity (number of stocks, expected
# returns, covariance/correlation matrix, prices, budget, the
# number of stocks to pick, and the penalty weights) is now
# read as input instead of being hard-coded. Press Enter on
# any prompt to fall back to the shown default value.


def ask(prompt, default, cast=float):
    """Prompt the user for a value, falling back to `default` on blank input."""
    raw = input(f"{prompt} [default: {default}]: ").strip()
    if raw == "":
        return default
    return cast(raw)


def ask_vector(prompt, n, default):
    """Prompt for n space-separated numbers; falls back to `default` list."""
    raw = input(f"{prompt} (space-separated, {n} values) "
                f"[default: {' '.join(map(str, default))}]: ").strip()
    if raw == "":
        return np.array(default, dtype=float)
    values = [float(v) for v in raw.split()]
    if len(values) != n:
        raise ValueError(f"Expected {n} values, got {len(values)}.")
    return np.array(values, dtype=float)


def ask_matrix(prompt, n, default):
    """Prompt for an n x n matrix, one row per line; falls back to `default`."""
    print(f"{prompt} (enter {n} rows, {n} space-separated numbers each; "
          f"blank row uses the default matrix)")
    rows = []
    for i in range(n):
        raw = input(f"  Row {i + 1}: ").strip()
        if raw == "":
            return np.array(default, dtype=float)
        values = [float(v) for v in raw.split()]
        if len(values) != n:
            raise ValueError(f"Row {i + 1}: expected {n} values, got {len(values)}.")
        rows.append(values)
    return np.array(rows, dtype=float)


def get_user_inputs():

    print("=== Portfolio QAOA Setup ===\n")

    n = ask("Number of stocks", 3, int)

    default_mu = [0.10, 0.15, 0.12][:n] + [0.10] * max(0, n - 3)
    default_prices = [100, 200, 150][:n] + [100] * max(0, n - 3)
    default_Sigma = np.eye(n) * 0.05  # fallback if n != 3

    print()
    mu = ask_vector("Expected returns", n, default_mu)

    print()
    if n == 3:
        default_cov = [
            [0.04, 0.01, 0.015],
            [0.01, 0.09, 0.02],
            [0.015, 0.02, 0.0625],
        ]
    else:
        default_cov = default_Sigma.tolist()
    Sigma = ask_matrix("Covariance matrix", n, default_cov)

    print()
    prices = ask_vector("Stock prices", n, default_prices)

    print()
    budget = ask("Total available budget", 250.0, float)
    K = ask("Number of stocks to select (K)", min(2, n), int)
    lam = ask("Risk-aversion parameter (lambda)", 1.0, float)
    P_K = ask("Penalty weight for the K-stocks constraint (P_K)", 1.0, float)
    P_B = ask("Penalty weight for the budget constraint (P_B)", 1.0, float)
    scale = ask("Scale factor to discretize prices/budget", 50.0, float)

    return n, mu, Sigma, prices, budget, K, lam, P_K, P_B, scale


# ============================================================
# 2. BUILD THE QUBO FOR AN ARBITRARY NUMBER OF STOCKS
# ============================================================
#
# Variables:
#   0 .. n-1        -> stock selection bits x_1 .. x_n
#   n .. n+m-1       -> slack bits s_0 .. s_(m-1), s = sum_k 2^k * s_k
#
# The number of slack qubits m is chosen automatically so that
# the slack variable can represent any integer from 0 up to the
# scaled budget B (m = ceil(log2(B + 1)), at least 0 if B == 0).

def build_qubo(n, mu, Sigma, p, B, K, lam, P_K, P_B):

    if B < 0:
        raise ValueError("Scaled budget must be non-negative.")

    n_slack = int(np.ceil(np.log2(B + 1))) if B >= 1 else 0
    n_qubits = n + n_slack

    linear = np.zeros(n_qubits)
    quadratic = {}
    constant = 0.0

    # ---- Return term: -mu^T x -----------------------------
    for i in range(n):
        linear[i] += -mu[i]

    # ---- Risk term: lambda * x^T Sigma x -------------------
    for i in range(n):
        linear[i] += lam * Sigma[i, i]
        for j in range(i + 1, n):
            coeff = lam * 2.0 * Sigma[i, j]
            quadratic[(i, j)] = quadratic.get((i, j), 0.0) + coeff

    # ---- Exactly-K-stocks constraint: P_K*(sum x_i - K)^2 --
    for i in range(n):
        linear[i] += P_K * (1 - 2 * K)
    for i in range(n):
        for j in range(i + 1, n):
            quadratic[(i, j)] = quadratic.get((i, j), 0.0) + P_K * 2
    constant += P_K * K ** 2

    # ---- Budget constraint: P_B*(p^T x + s - B)^2 ----------
    # budget_coeff holds the coefficient of every qubit (stocks
    # first, then the binary-weighted slack bits).
    budget_coeff = np.concatenate([
        p,
        np.array([2 ** k for k in range(n_slack)], dtype=float)
    ])

    for i in range(n_qubits):
        a = budget_coeff[i]
        linear[i] += P_B * (a ** 2 - 2 * B * a)

    for i in range(n_qubits):
        for j in range(i + 1, n_qubits):
            coeff = P_B * 2 * budget_coeff[i] * budget_coeff[j]
            quadratic[(i, j)] = quadratic.get((i, j), 0.0) + coeff

    constant += P_B * B ** 2

    return linear, quadratic, constant, n_qubits, n_slack, budget_coeff


# ============================================================
# 3. CONVERT QUBO -> ISING HAMILTONIAN
# ============================================================

def build_hamiltonian(linear, quadratic, constant, n_qubits):

    pauli_terms = {}
    hamiltonian_constant = constant

    def add_pauli(pauli_string, coefficient):
        pauli_terms[pauli_string] = pauli_terms.get(pauli_string, 0.0) + coefficient

    # Linear terms
    for i in range(n_qubits):
        a = linear[i]
        hamiltonian_constant += a / 2

        pauli = ["I"] * n_qubits
        pauli[i] = "Z"
        add_pauli("".join(pauli), -a / 2)

    # Quadratic terms
    for (i, j), b in quadratic.items():
        hamiltonian_constant += b / 4

        pauli_i = ["I"] * n_qubits
        pauli_i[i] = "Z"
        add_pauli("".join(pauli_i), -b / 4)

        pauli_j = ["I"] * n_qubits
        pauli_j[j] = "Z"
        add_pauli("".join(pauli_j), -b / 4)

        pauli_ij = ["I"] * n_qubits
        pauli_ij[i] = "Z"
        pauli_ij[j] = "Z"
        add_pauli("".join(pauli_ij), b / 4)

    hamiltonian_list = [
        (pauli_string, coeff)
        for pauli_string, coeff in pauli_terms.items()
        if abs(coeff) > 1e-12
    ]

    H_cost = SparsePauliOp.from_list(hamiltonian_list)

    return H_cost, pauli_terms, hamiltonian_constant


# ============================================================
# 4. CLASSICAL BRUTE-FORCE CHECK (arbitrary n)
# ============================================================

def brute_force_check(n, mu, Sigma, prices, budget, K):

    print("\n================ VALID PORTFOLIOS ================")

    best_cost = np.inf
    best_portfolio = None

    for combo in itertools.product([0, 1], repeat=n):

        x = np.array(combo)

        if np.sum(x) != K:
            continue

        money = np.dot(prices, x)
        if money > budget:
            continue

        return_value = np.dot(mu, x)
        risk = x.T @ Sigma @ x
        cost = -return_value + risk

        print(
            f"{x}  "
            f"cost = {cost:.6f}, "
            f"return = {return_value:.4f}, "
            f"risk = {risk:.6f}, "
            f"budget = {money}"
        )

        if cost < best_cost:
            best_cost = cost
            best_portfolio = x.copy()

    print("\nBest classical portfolio:")
    print(best_portfolio)
    print("Best cost:", best_cost)

    return best_portfolio, best_cost


# ============================================================
# 5. QAOA CIRCUIT (mixer + cost unitary), arbitrary n_qubits
# ============================================================

def apply_mixer(qc, beta, n_qubits):
    for q in range(n_qubits):
        qc.rx(2 * beta, q)


def apply_cost_unitary(qc, gamma, H_cost):

    labels = H_cost.paulis.to_labels()
    coeffs = H_cost.coeffs

    # Single-qubit Z terms
    for pauli_string, coefficient in zip(labels, coeffs):
        z_positions = [i for i, p in enumerate(pauli_string) if p == "Z"]
        if len(z_positions) == 1:
            q = z_positions[0]
            qc.rz(2 * gamma * coefficient.real, q)

    # Two-qubit ZZ terms
    for pauli_string, coefficient in zip(labels, coeffs):
        z_positions = [i for i, p in enumerate(pauli_string) if p == "Z"]
        if len(z_positions) == 2:
            q1, q2 = z_positions
            qc.cx(q1, q2)
            qc.rz(2 * gamma * coefficient.real, q2)
            qc.cx(q1, q2)


def build_qaoa_circuit(n_qubits, H_cost):

    gamma = Parameter("γ")
    beta = Parameter("β")

    qc = QuantumCircuit(n_qubits)

    for q in range(n_qubits):
        qc.h(q)

    apply_cost_unitary(qc, gamma, H_cost)
    apply_mixer(qc, beta, n_qubits)

    return qc, gamma, beta


# ============================================================
# 6. NUMERICAL OPTIMIZATION
# ============================================================

def make_expectation_value(qc, gamma, beta, n_qubits, constant, linear,
                            quadratic, simulator, shots=4096):

    def expectation_value(params):

        gamma_value, beta_value = params

        circuit = qc.assign_parameters({gamma: gamma_value, beta: beta_value})

        measured = circuit.copy()
        measured.measure_all()

        result = simulator.run(measured, shots=shots).result()
        counts = result.get_counts()

        expectation = 0.0

        for bitstring, count in counts.items():

            bits = bitstring.replace(" ", "")[::-1]  # Qiskit bit order is reversed

            binary = np.array([int(bit) for bit in bits])
            probability = count / shots

            qubo_energy = constant
            qubo_energy += np.dot(linear, binary)
            for (i, j), coeff in quadratic.items():
                qubo_energy += coeff * binary[i] * binary[j]

            expectation += probability * qubo_energy

        return expectation

    return expectation_value


def optimize_qaoa(expectation_value, initial_points=None, maxiter=50):

    if initial_points is None:
        initial_points = [
            [0.1, 0.1],
            [0.5, 0.2],
            [1.0, 0.5],
            [1.5, 1.0],
        ]

    best_result = None

    for initial in initial_points:

        result = minimize(
            expectation_value,
            initial,
            method="COBYLA",
            options={"maxiter": maxiter},
        )

        if best_result is None or result.fun < best_result.fun:
            best_result = result

    return best_result


# ============================================================
# 7. DECODE THE MOST PROBABLE STATE
# ============================================================

def decode_solution(counts, n, n_slack, prices, budget):

    sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)

    print("\n================ MEASUREMENT ================")
    for bitstring, count in sorted_counts[:15]:
        print(bitstring, " : ", count)

    best_bitstring = sorted_counts[0][0]
    bits = best_bitstring.replace(" ", "")[::-1]
    binary = np.array([int(bit) for bit in bits])

    stocks = binary[:n]
    slack_bits = binary[n:n + n_slack]
    slack = sum(2 ** k * b for k, b in enumerate(slack_bits))

    print("\n================ SOLUTION ================")
    print("Stock bits:", stocks)
    print("Slack bits:", list(slack_bits))
    print("Slack:", slack)
    print("Selected stocks:", [i + 1 for i in range(n) if stocks[i] == 1])
    print("Actual budget used:", np.dot(prices, stocks))
    print("Budget:", budget)

    return stocks, slack


# ============================================================
# 8. MAIN
# ============================================================

def main():

    n, mu, Sigma, prices, budget, K, lam, P_K, P_B, scale = get_user_inputs()

    # ---- Scale prices/budget so they become small integers ----
    p = np.floor(prices / scale)
    B = np.floor(budget / scale)

    print("\nScaled prices:", p)
    print("Scaled budget:", B)

    # ---- Build QUBO ----
    linear, quadratic, constant, n_qubits, n_slack, budget_coeff = build_qubo(
        n, mu, Sigma, p, B, K, lam, P_K, P_B
    )

    print("\n================ QUBO ================")
    print("Constant =", constant)
    print("\nLinear coefficients:")
    for i, value in enumerate(linear):
        print(f"x[{i}] : {value:.6f}")
    print("\nQuadratic coefficients:")
    for (i, j), value in quadratic.items():
        print(f"x[{i}] x[{j}] : {value:.6f}")
    print(f"\nTotal qubits: {n_qubits} ({n} stock + {n_slack} slack)")

    # ---- Build Hamiltonian ----
    H_cost, pauli_terms, hamiltonian_constant = build_hamiltonian(
        linear, quadratic, constant, n_qubits
    )

    print("\n================ HAMILTONIAN ================")
    print(H_cost)
    print("\nHamiltonian constant =", hamiltonian_constant)

    # ---- Classical brute-force check (uses the *unscaled* data) ----
    brute_force_check(n, mu, Sigma, prices, budget, K)

    # ---- Build QAOA circuit ----
    qc, gamma, beta = build_qaoa_circuit(n_qubits, H_cost)

    print("\n================ QAOA CIRCUIT ================")
    print(qc.draw())

    # ---- Optimize ----
    simulator = AerSimulator()

    expectation_value = make_expectation_value(
        qc, gamma, beta, n_qubits, constant, linear, quadratic, simulator
    )

    best_result = optimize_qaoa(expectation_value)

    print("\n================ OPTIMIZATION ================")
    print("Optimal gamma =", best_result.x[0])
    print("Optimal beta  =", best_result.x[1])
    print("Expectation   =", best_result.fun)

    # ---- Final measurement ----
    final_circuit = qc.assign_parameters({
        gamma: best_result.x[0],
        beta: best_result.x[1],
    })
    final_circuit.measure_all()

    result = simulator.run(final_circuit, shots=10000).result()
    counts = result.get_counts()

    decode_solution(counts, n, n_slack, prices, budget)


if __name__ == "__main__":
    main()

  Using cached rustworkx-0.18.1-cp310-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (10 kB)
  Using cached stevedore-5.9.1-py3-none-any.whl.metadata (2.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 4.2 MB/s eta 0:00:00
=== Portfolio QAOA Setup ===

Number of stocks [default: 3]: 

Expected returns (space-separated, 3 values) [default: 0.1 0.15 0.12]: 

Covariance matrix (enter 3 rows, 3 space-separated numbers each; blank row uses the default matrix)
  Row 1: 

Stock prices (space-separated, 3 values) [default: 100 200 150]: 

Total available budget [default: 250.0]: 
Number of stocks to select (K) [default: 2]: 
Risk-aversion parameter (lambda) [default: 1.0]: 
Penalty weight for the K-stocks constraint (P_K) [default: 1.0]: